🔹 STEP 1: Install Dependencies

In [1]:
!pip install agno groq ddgs gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 29.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.3/142.3 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.0/67.0 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 59.5 MB/s eta 0:00:00


🔹 STEP 2: Suppress Warnings (Optional)

In [2]:
import warnings
warnings.filterwarnings("ignore")

🔹 STEP 3: Set API Key

In [3]:
import os
os.environ["GROQ_API_KEY"] = "Add_Your_API_Key"

🔹 STEP 4: Imports

In [4]:
from agno.agent import Agent
from agno.models.groq import Groq
from ddgs import DDGS
import gradio as gr

🔹 STEP 5: Search Tool

In [5]:
def search_tool(query: str) -> str:
    results = []

    with DDGS() as ddgs:
        for r in ddgs.text(query, max_results=5):
            if "body" in r and len(r["body"]) > 50:
                results.append(r["body"])

    return "\n".join(results)

🔹 STEP 6: Agents

🔍 Research Agent



In [6]:
research_agent = Agent(
    name="Travel Research Agent",
    model=Groq(id="llama-3.1-8b-instant"),
    tools=[search_tool],
    instructions="""
    You are a travel researcher.
    Find key insights:
    - Attractions
    - Budget stays
    - Food
    - Tips
    Keep output concise.
    """
)

🧠 Summarizer Agent

In [7]:
summary_agent = Agent(
    name="Summarizer Agent",
    model=Groq(id="llama-3.1-8b-instant"),
    instructions="""
    Summarize the research into short bullet points.
    Keep it under 200 words.
    """
)

🧭 Planner Agent

In [8]:
planner_agent = Agent(
    name="Travel Planner Agent",
    model=Groq(id="llama-3.1-8b-instant"),
    instructions="""
    Create:
    - Day-wise itinerary
    - Budget
    - Food & transport tips
    Keep it realistic.
    """
)

🧠 Reviewer Agent

In [9]:
review_agent = Agent(
    name="Travel Plan Reviewer",
    model=Groq(id="llama-3.1-8b-instant"),
    instructions="""
    Improve the plan:
    - Fix timing
    - Optimize budget
    - Make it practical
    Return final plan.
    """
)

🔹 STEP 7: Workflow Function (IMPORTANT)

In [28]:
def travel_planner(user_query):

    # Step 1
    research_data = research_agent.run(user_query)
    research_text = research_data.content

    # Step 2
    summary = summary_agent.run(research_text[:2000])
    summary_text = summary.content

    # Step 3
    draft_plan = planner_agent.run(f"""
    USER REQUEST:
    {user_query}

    RESEARCH SUMMARY:
    {summary_text[:1000]}
    """)

    draft_text = draft_plan.content

    # Step 4
    final_plan = review_agent.run(draft_text[:1500])

    return final_plan.content

🔹 STEP 8: Test (Optional)

In [21]:
#print(travel_planner("Plan a 3-day Goa trip for beaches and nightlife"))

🔹 STEP 9: Gradio Wrapper

In [29]:
def gradio_travel_planner(user_query):
    return travel_planner(user_query)

🔹 STEP 10: Build Gradio UI

In [30]:
interface = gr.Interface(
    fn=gradio_travel_planner,
    inputs=gr.Textbox(
        lines=2,
        placeholder="e.g. Plan a 3-day Goa trip for beaches and nightlife"
    ),
    outputs=gr.Textbox(label="Travel Plan", lines=20),
    title="🌍 AI Travel Planner",
    description="Agentic AI Travel Planner using Groq + Agno"
)

🔹 STEP 11: Launch App

In [31]:
interface.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://f98fefc5670e19f0ff.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


🔹 STEP 12: Launch App For Interactive UI

In [15]:
import gradio as gr

def gradio_travel_planner(user_query):
    return travel_planner(user_query)

with gr.Blocks(theme=gr.themes.Soft()) as demo:

    gr.Markdown("# 🌍 AI Travel Planner")
    gr.Markdown("### Plan your perfect trip using Agentic AI (Groq + Agno)")

    with gr.Row():
        with gr.Column():
            user_input = gr.Textbox(
                label="✈️ Enter your travel query",
                placeholder="e.g. Plan a 5-day trip to Manali with adventure and budget ₹20k",
                lines=3
            )

            submit_btn = gr.Button("🚀 Generate Plan")
            clear_btn = gr.Button("🧹 Clear")

        with gr.Column():
            output = gr.Markdown(label="📍 Travel Plan")

    # Actions
    submit_btn.click(fn=gradio_travel_planner, inputs=user_input, outputs=output)
    clear_btn.click(lambda: "", None, output)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://6d5186646a51355269.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
